# Introduction to Modules

POPSIM is not a single simulator. Rather, it is a simulation toolkit that empowers you to define "modules", where each module itself is a simulator that can be run on its own using the same set of tools. To create more complex simulators, one can create _composite modules_ by simply creating a new module that calls other modules. You will learn more about composite modules in [a later tutorial](./controller_plus_sim.ipynb)

## Implementing a Basic Module: The Lorenz System
Let's proceed with a basic example, using both math and code. Let's say we want to perform a simulation of the Lorenz system:

$$\begin{aligned}
\frac{dx}{dt} &= \sigma(y - x)  \\
\frac{dy}{dt} &= x(\rho - z) - y  \\
\frac{dz}{dt} &= xy - \beta z\\
\end{aligned}$$

In addition, we are given the following tasks:

1. Allow for the simulation inputs, such as $\rho$, to change over time.
   
2. Compute the distance of the state from the origin.
   
3. Freeze the simulation if the simulation goes out of bounds.

To achieve this task, let's first break-down the problem by defining four categories of variables involved in this problem:

1. State Variables $\mathbf{x}(t)$: These are the variables that define the state of the system and are evolved over time. In this problem, $\mathbf{x} = [x, y, z]$.
   
2. Input Variables $\mathbf{p}(t)$: These are the variables that are possibly time-dependent and are prescribed. In this problem, $\mathbf{p}(t) = [\rho(t), \sigma(t), \beta(t)]$. For example, Lorenz, in his 1963 paper, has an example where $\sigma=10$, $\beta=8/3$, and $\rho=28$. Note that time-independent inputs can also be thought of as a special case of time-dependent inputs.
   
3. Output Variables $\mathbf{o}(t)$: These are the variables that are computed from the state and input variables. That is, there is some function $\mathcal{O}$ for which $\mathbf{o}(t) = \mathcal{O}(\mathbf{x}(t), \mathbf{p}(t))$. In this problem, $\mathbf{o}(t) = [d(t)]$, where $d(t)$ is the distance of the state from the origin.
   
4. Configuration Variables $\mathcal{C}$: These are the variables that are used to configure the simulation and are static across time. Note that the key distinction between putting time-independent inputs in $\mathcal{C}$ instead of $\mathbf{p}$ is that you do not have the option of making the inputs time-dependent. In this problem, the configuration variables are the state limits $\mathcal{C} = (\mathbf{x}_{min}, \mathbf{x}_{max})$.


With these definitions in place, we can mathematically define a module $\mathcal{M}_\mathcal{C}$ parameterized by a configuration $\mathcal{C}$ as follows:

$$\dot{\mathbf{x}}(t), \mathbf{o}(t) = \mathcal{M}_\mathcal{C}(\mathbf{x}(t), \mathbf{p}(t))$$

For individuals with a control systems or reinforcement learning (RL) background, the above should remind you of a state-space system or the dynamics model of a Partially Observable Markov Decision Process (POMDP).

Now let's take a look at a code example that implements this module.

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect

from IPython.display import Markdown, display

from popsim.modules.module_examples import BasicLorenz

display(Markdown(f"```python\n{inspect.getsource(BasicLorenz)}\n```"))

### The init-call Pattern
A pretty common (and useful) pattern with modules is to adopt an "init-call" structure, where the `__init__` function does the job of initializing and setting up the module (in this example, setup isn't really needed). The `__call__` function is then the implementation of the module itself. Notice how the `__call__` function is essentially the software realization of the mathematical definition of the module:

$$\dot{\mathbf{x}}(t), \mathbf{o}(t) = \mathcal{M}_\mathcal{C}(\mathbf{x}(t), \mathbf{p}(t))$$

### Simulating the Module
Now to simulate the module, we just need to spin up the module and define the time base, initial state, and inputs. Note that we get back the state and output variables for each time slice in the time base, so we can make a plot of the state trajectory in addition to a plot of the distance from the origin.

In [ ]:
import holoviews as hv

hv.extension("matplotlib")

from popsim.simulate import make_time_base, simulate, SimInput

# Define the module, time_base, initial_state, and inputs.
module = BasicLorenz(config=BasicLorenz.Config())
time_base = make_time_base(t0=0.0, t1=3.0, dt=0.01)
initial_state = BasicLorenz.State(x=1.0, y=1.0, z=1.0)
inputs = BasicLorenz.Inputs(sigma=10.0, rho=28.0, beta=8.0 / 3.0)

ds = simulate(module=module, sim_inputs=SimInput(time=time_base, initial_state=initial_state, inputs=inputs), return_xarray=True)

print(ds)

hv.Scatter3D((ds["state.x"], ds["state.y"], ds["state.z"])) + hv.Curve(ds["output.distance_from_origin"])

### Time-Dependent Inputs
We can now specify another simulation where there is a time-varying $\rho(t)$ and observe how the system evolves.

In [ ]:
inputs_time_dep_rho = BasicLorenz.Inputs(sigma=10.0, rho={0.0: 28.0, 1.0: 10.0, 2.0: 1.0}, beta=8.0 / 3.0)
sim_input = SimInput(time=time_base, initial_state=initial_state, inputs=inputs_time_dep_rho)
ds_rho = simulate(module=module, sim_inputs=sim_input, return_xarray=True)

# Overlay the two simulations.
hv.Scatter3D((ds["state.x"], ds["state.y"], ds["state.z"])) * hv.Scatter3D((ds_rho["state.x"], ds_rho["state.y"], ds_rho["state.z"]))

### Testing the Out-of-Bounds Condition
Finally, we can test the out-of-bounds condition by setting the y limits to be intentionally too narrow and observing how the simulation behaves.

In [ ]:
config = BasicLorenz.Config()
config.ylims = (0.0, 5.0)
module = BasicLorenz(config=config)

ds = simulate(module=module, sim_inputs=SimInput(time=time_base, initial_state=initial_state, inputs=inputs), return_xarray=True)

hv.Scatter3D((ds["state.x"], ds["state.y"], ds["state.z"])) + hv.Curve(ds["output.distance_from_origin"])

## Discrete-Time Modules

In our basic example, the module is behaving as a continuous-time system. However, there are often cases where state variables can't be described as continuous-time variables. For example, perhaps we want to define a state variable for whether the plasma is disrupted or not:

In [ ]:
from popsim.modules.module_examples import ExampleDisruptedState

display(Markdown(f"```python\n{inspect.getsource(ExampleDisruptedState)}\n```"))

Now let's say we want to cook up a disruption detector module that takes in a disruptivity and decides whether to make the plasma disrupted based on that. That is:

1. If disruptivity exceeds some threshold, the plasma becomes disrupted.
2. If the plasma is already disrupted, but disruptivity falls below some threshold, the plasma still stays disrupted.

This is a basic example of a state machine. While it is possible to implement this as a continuous-time differential equation, it is perhaps more natural to make a discrete jump from`NOT_DISRUPTED` to `DISRUPTED`. That is, we would like to go from a mathematical model of the form on the left to the form on the right:

$$\dot{\mathbf{x}}(t), \mathbf{o}(t) = \mathcal{M}_\mathcal{C}(\mathbf{x}(t), \mathbf{p}(t)) \Rightarrow \mathbf{x}_{t+1}, \mathbf{o}_t = \mathcal{M}_\mathcal{C}(\mathbf{x}_t, \mathbf{p}_t)$$

To mark certain variables as having the relationship on the right hand side, we can use `popsim.discrete_time_field`. The below module provides an example of how you would use it. Once you have labelled a variable as a discrete-time field, you can use the `__call__` function to update it from time step to time step.

In [ ]:
from popsim.modules.module_examples import DiscreteTimeExample

display(Markdown(f"```python\n{inspect.getsource(DiscreteTimeExample)}\n```"))

### Simulating the Module
Okay, now that we have the module, let's try ramping up the disruptivity to 1 and back down again and see how the plasma disruption state changes.

In [ ]:
import hvplot.xarray  # noqa: F401

hvplot.extension("bokeh")

time_base = make_time_base(t0=0.0, t1=1.0, dt=0.01)
module = DiscreteTimeExample(config=DiscreteTimeExample.Config(disruptivity_threshold=0.9))
initial_state = DiscreteTimeExample.State(disrupted_state=ExampleDisruptedState.NOT_DISRUPTED)
inputs = DiscreteTimeExample.Inputs(disruptivity={0.0: 0.1, 0.5: 1.0, 0.75: 0.0})  # Time-dependent disruptivity.
sim_input = SimInput(time=time_base, initial_state=initial_state, inputs=inputs)
ds = simulate(module=module, sim_inputs=sim_input, return_xarray=True)

ds.hvplot()

## Mixed Discrete-Time and Continuous-Time (Hybrid) Modules
What if the system dynamics have both continuous-time and discrete-time components? Such is the case for many real-world systems, especially if you want to simulate not just the physics, but also hardware details and the control system itself. Such systems are often referred to as **hybrid systems** in the control literature.

The mathematical notation is going to get a bit burdensome, but the practical way you handle it in POPSIM is simple.
> **_It's Easy:_**  just mark discrete-time variables with `discrete_time_field`. Everything else is continuous-time.

Let's consider the example of a pong-like situation, where the ball is bouncing back and forth between two boundaries at a constant speed. However, it's velocity is determined by the direction it is going, and when it hits a boundary, it reverses direction. This is a hybrid system because the velocity is continuous-time, but the position is discrete-time. Such a system is implemented below.

In [ ]:
from popsim.modules.module_examples import HybridExample

display(Markdown(f"```python\n{inspect.getsource(HybridExample)}\n```"))

> **_It's a Bit Confusing:_**  In the pure continuous case, the output state structure was all time derivatives, and in the pure discrete case, the output state structure was always the state a the next time step, but in the hybrid case, it ends up being both in the same structure. Continuous variables are time derivatives, and discrete variables are the state at the next time step.

In [ ]:
module = HybridExample()
initial_state = HybridExample.State(y=0.0, sign=1)
inputs = HybridExample.Inputs(speed=1.0, ylims=({0.0: -1.0, 10.0: -0.01}, {0.0: 1.0, 10.0: 0.01}))
time_base = make_time_base(t0=0.0, t1=10.0, dt=0.001)
sim_input = SimInput(time=time_base, initial_state=initial_state, inputs=inputs)
ds = simulate(module=module, sim_inputs=sim_input, return_xarray=True)

(ds.hvplot.line(y=["inputs.ylims.0", "inputs.ylims.1", "state.y"]) + ds.hvplot.line(y="state.sign"))